# Digital Echoes vs. Bed Nights
## Predicting Viral Tourism Demand Shocks from Digital Signals

**DTU 42578 Advanced Business Analytics — Spring 2026**  
*Group: Alex (Spain · ES), Beatriz (Croatia · HR), Cesia (Italy · IT & Albania · AL), Frede (Malta · MT & Data Engineering)*

---

## 1. Introduction

Viral social-media content can trigger rapid, hard-to-predict surges in tourism demand — so-called *viral tourism shocks*. This report investigates whether **digital attention signals** (Google Trends, Wikipedia pageviews) measured *before* a surge carry sufficient predictive power to forecast Eurostat bed-night arrivals at the regional level across six EU destinations: Spain, Croatia, Italy, Albania, Malta, and Greece.

**Research question:** Do digital signals predict tourist arrivals with a measurable lead time, and can we build a resilience-aware system to help destinations prepare?

**Methods (≥ 3 course topics):**
| Method | Course topic |
|--------|-------------|
| Random Forest + SHAP | Forecasting · Explainable AI |
| Graph Neural Network (GNN) | GNNs — spatial spillover |
| Recommender System | Recommender systems |
| Granger causality + COVID policy controls | Causal methods · Endogeneity |
| Seasonality decomposition (month + YoY) | Time-series methods |

---
## 2. Data Collection

### 2.0 Setup & Imports

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 120, 'figure.figsize': (12, 4)})

# ── Clone repo (skips if already present) ────────────────────────────────────
if not os.path.exists('viral-tourism-resilience'):
    os.system('git clone https://github.com/cniq182/viral-tourism-resilience.git')

DATA_DIR    = 'viral-tourism-resilience/data/processed'
MASTER_FILE = os.path.join(DATA_DIR, 'master_tourism_dataset.csv')
OXCGRT_FILE = os.path.join(DATA_DIR, 'oxcgrt_covid_data.csv')

print('Master file exists:', os.path.exists(MASTER_FILE))


### 2.1 Datasets Used

| Dataset | Source | Coverage | Key variables |
|---------|--------|----------|---------------|
| Eurostat tourist nights spent (tour_occ_nin2m) | Eurostat (Excel download) | EU NUTS2 regions, 2020–2026 | `nights_spent_nuts2`, `nights_spent_country` |
| Eurostat monthly arrivals (tour_occ_arm) | Eurostat API | Country level, 2020–2026 | `arrivals_country` |
| Eurostat bed places (tour_cap_nat) | Eurostat API | Country level, annual | `bed_places_country` |
| Google Trends | pytrends (unofficial API) | 30 cities, worldwide normalisation, 2020–2026 | `gt_airbnb`, `gt_hotel`, `gt_flights`, `gt_attraction_1/2/3` |
| OxCGRT COVID policy (v4) | Oxford COVID-19 Government Response Tracker | Country-level, 2020–2022 | `C6M_stay_at_home`, `C4M_gathering_restrictions`, `C8EV_travel_controls` |

**Destinations covered:**  
Spain (ES — 6 cities, 6 NUTS2 regions), Croatia (HR — 5 cities, 4 NUTS2 regions), Italy (IT — 5 cities, 5 NUTS2 regions), Albania (AL — 5 cities, 3 NUTS2 regions), Malta (MT — 4 cities, **1 NUTS2 region**), Greece (GR — 5 cities, 5 NUTS2 regions).  
**Total: 2,250 rows × 16 columns** (75 months × 30 cities).


### 2.2 APIs & Data Acquisition

**Google Trends — pytrends**  
Data collected via `pytrends.TrendReq` with worldwide geo-filter (`geo=''`) to ensure comparability across countries. Queries batched in groups of 5 with an anchor term for cross-batch normalisation. *Note: urllib3 v2 requires a monkey-patch (`method_whitelist` removal) before importing `TrendReq`.*

**Eurostat — SDMX REST API**  
Dataset `tour_occ_nin2m` (monthly nights spent by residents and non-residents, NUTS2 level). Downloaded as TSV and reshaped via `cleaning_data.py`.

**Wikipedia — Wikimedia REST API**  
`https://wikimedia.org/api/rest_v1/metrics/pageviews/per-article/…` — daily pageviews aggregated to monthly.

**OxCGRT — direct CSV download**  
Oxford's `OxCGRT_compact_national_v1.csv` filtered to indicators C4M, C6M, C8EV and aggregated to monthly means.

### 2.3 Data Pipeline Architecture

> **TODO (Frede):** Insert architecture diagram here (the drawing you made showing the full pipeline from raw APIs → master dataset → feature engineering → models).

<!-- Embed as: ![Pipeline architecture](images/architecture.png) -->

### 2.4 Final Dataset

In [ ]:
df = pd.read_csv(MASTER_FILE, parse_dates=['date'])
df.sort_values(['country', 'region', 'city', 'date'], inplace=True)
df.reset_index(drop=True, inplace=True)

print(f'Shape: {df.shape}')
print(f'Countries: {sorted(df["country"].unique())}')
print(f'Date range: {df["date"].min().date()} → {df["date"].max().date()}')
print(f'Columns: {list(df.columns)}')
df.head()


---
## 3. Dataset Pre-processing & Visualisation

### 3.1 Seasonality Features

Three features capture the seasonal structure of tourism demand:  
- `month` — calendar month (1–12) used as a cyclic control  
- `arrivals_last_year` — 12-month lag of `arrivals_country` within each geographic unit  
- `arrivals_yoy_pct_change` — year-over-year percentage change  

*Script adapted from `FINAL/add_seasonality.py` (Frede's branch).*

In [ ]:
GEO_KEYS = ['country', 'region', 'city']

# Month
df['month'] = df['date'].dt.month

# 12-month lag within each geographic unit
df['arrivals_last_year'] = (
    df.groupby(GEO_KEYS)['arrivals_country'].shift(12)
)

# Year-over-year % change
df['arrivals_yoy_pct_change'] = (
    (df['arrivals_country'] - df['arrivals_last_year'])
    / df['arrivals_last_year'].replace(0, np.nan)
    * 100
)

print('Seasonality features added.')
print(df[['date', 'country', 'city', 'month', 'arrivals_country',
          'arrivals_last_year', 'arrivals_yoy_pct_change']].dropna().head(10))

### 3.2 Endogeneity Controls — COVID Policy Stringency

To address endogeneity between digital signals and actual arrivals during the pandemic, we merge three OxCGRT indicators as control variables:

| Feature | OxCGRT indicator | Interpretation |
|---------|-----------------|----------------|
| `dest_stay_at_home` | C6M | Destination stay-at-home stringency |
| `dest_gathering_restrictions` | C4M | Restrictions on gatherings at destination |
| `origin_top3_travel_ban_avg` | C8EV (avg. top-3 origin countries) | International travel ban from main sending markets |

*Script adapted from `FINAL/build_dataset_custom_covid.py` (Frede's branch).*

In [ ]:
import os

TOP_ORIGINS = {
    'Albania':  ['Kosovo', 'North Macedonia', 'Italy'],
    'Croatia':  ['Germany', 'Slovenia', 'Austria'],
    'Greece':   ['United Kingdom', 'Germany', 'France'],
    'Italy':    ['Germany', 'United States', 'United Kingdom'],
    'Malta':    ['United Kingdom', 'Italy', 'France'],
    'Spain':    ['United Kingdom', 'France', 'Germany'],
}

if not os.path.exists(OXCGRT_FILE):
    print(f'[SKIP] OxCGRT file not found at {OXCGRT_FILE}. '
          'Download oxcgrt_covid_data.csv and place it in DATA_DIR to run this section.')
else:
    ox = pd.read_csv(
        OXCGRT_FILE,
        usecols=['CountryName', 'Date',
                 'C8EV_International travel controls',
                 'C6M_Stay at home requirements',
                 'C4M_Restrictions on gatherings'],
        low_memory=False,
    )
    ox['Date'] = pd.to_datetime(ox['Date'].astype(str), format='%Y%m%d')
    ox['year_month'] = ox['Date'].values.astype('datetime64[M]')
    ox_m = ox.groupby(['CountryName', 'year_month'], as_index=False).mean(numeric_only=True)

    # Destination controls
    dest_ox = ox_m[['CountryName', 'year_month',
                    'C6M_Stay at home requirements',
                    'C4M_Restrictions on gatherings']].rename(columns={
        'C6M_Stay at home requirements':  'dest_stay_at_home',
        'C4M_Restrictions on gatherings': 'dest_gathering_restrictions',
    })
    df = df.merge(dest_ox, how='left',
                  left_on=['country', 'date'],
                  right_on=['CountryName', 'year_month'])
    df.drop(columns=['CountryName', 'year_month'], inplace=True)

    # Origin travel-ban average (top 3 sending countries)
    travel_lkp = ox_m.set_index(['CountryName', 'year_month'])[
        'C8EV_International travel controls'].to_dict()

    def _origin_ban_avg(row):
        origins = TOP_ORIGINS.get(row['country'], [])
        vals = [travel_lkp.get((o, row['date'])) for o in origins]
        vals = [v for v in vals if v is not None]
        return sum(vals) / len(vals) if vals else np.nan

    df['origin_top3_travel_ban_avg'] = df.apply(_origin_ban_avg, axis=1)

    # Fill pre/post-pandemic NaNs with 0
    covid_cols = ['dest_stay_at_home', 'dest_gathering_restrictions', 'origin_top3_travel_ban_avg']
    df[covid_cols] = df[covid_cols].fillna(0)

    print('COVID policy controls merged.')
    print(df[['date', 'country', 'city'] + covid_cols].dropna().head(8))

### 3.3 Exploratory Data Analysis & Visualisation

> Each team member adds plots for their own region(s) below.

In [ ]:
# ── Spain — EDA (Alex) ────────────────────────────────────────────────────────
# TODO (Alex): time-series plot of GT signals vs. nights_spent for ES regions,
# shock detection markers (>2σ above 3yr rolling mean), correlation heatmap.
pass

In [ ]:
# ── Croatia — EDA (Beatriz) ───────────────────────────────────────────────────
# TODO (Beatriz): time-series, shock markers, correlation heatmap for HR.
pass

In [ ]:
# ── Italy — EDA (Cesia) ───────────────────────────────────────────────────────
# TODO (Cesia): time-series, shock markers, correlation heatmap for IT.
pass

In [ ]:
# ── Albania — EDA (Cesia) ─────────────────────────────────────────────────────
# TODO (Cesia): time-series, shock markers, correlation heatmap for AL.
pass

In [ ]:
# ── Malta — EDA (Frede) ───────────────────────────────────────────────────
# Malta has 4 cities but ONE NUTS2 region — aggregate before plotting.
MT_GT_COLS = ['gt_airbnb','gt_hotel','gt_flights','gt_attraction_1','gt_attraction_2','gt_attraction_3']
df_malta_region = (
    df[df['country']=='Malta']
    .groupby('date', as_index=False)
    .agg({**{c: 'mean' for c in MT_GT_COLS},
          'nights_spent_nuts2': 'first',
          'arrivals_country':   'first',
          'month': 'first'})
)
# TODO (Frede): time-series plot of GT signals vs. nights_spent_nuts2 for Malta,
# shock detection markers (>2σ above 3yr rolling mean), correlation heatmap.
pass


In [ ]:
# ── Greece — EDA (Alex / Beatriz) ────────────────────────────────────────
# TODO: time-series plot of GT signals vs. nights_spent_nuts2 for GR cities,
# shock detection markers (>2σ above 3yr rolling mean), correlation heatmap.
pass


---
## 4. Prediction

### 4.1 Why Random Forest?

> **TODO (all):** Justify the model choice here — why Random Forest over linear regression, ARIMA, or gradient boosting for this setting? Consider: non-linearity, mixed feature types (GT signals + seasonal + policy controls), interpretability via SHAP, robustness to missing values.

### 4.2 Feature Engineering


In [ ]:
GT_COLS      = ['gt_airbnb','gt_hotel','gt_flights',
                'gt_attraction_1','gt_attraction_2','gt_attraction_3']
COVID_COLS   = ['dest_stringency_index','origin_top3_stringency_avg']
SEASON_COLS  = ['month','arrivals_last_year','arrivals_yoy_pct_change']
FEATURE_COLS = GT_COLS + COVID_COLS + SEASON_COLS
TARGET       = 'nights_spent_nuts2'

# Malta: aggregate 4 cities → 1 region before modelling
malta_agg = (
    df[df['country']=='Malta']
    .groupby('date', as_index=False)
    .agg({**{c: 'mean' for c in GT_COLS + COVID_COLS},
          **{c: 'first' for c in SEASON_COLS + [TARGET,'country','region']}})
)
malta_agg['city'] = 'malta'

df_model = pd.concat([
    df[df['country'] != 'Malta'],
    malta_agg
], ignore_index=True)

# One-hot encode country to let the model distinguish destinations
df_model = pd.get_dummies(df_model, columns=['country'], drop_first=False)
country_dummies = [c for c in df_model.columns if c.startswith('country_')]
FEATURE_COLS_FULL = FEATURE_COLS + country_dummies

df_model = df_model.dropna(subset=FEATURE_COLS_FULL + [TARGET])
print(f'Model dataset: {df_model.shape[0]} rows, {len(FEATURE_COLS_FULL)} features')
print('Features:', FEATURE_COLS_FULL)


### 4.3 Train / Test Split


In [ ]:
# Time-based split: last 12 months = test (2025-04 → 2026-03)
cutoff = df_model['date'].max() - pd.DateOffset(months=12)
train  = df_model[df_model['date'] <= cutoff]
test   = df_model[df_model['date'] >  cutoff]

X_train = train[FEATURE_COLS_FULL]
y_train = train[TARGET]
X_test  = test[FEATURE_COLS_FULL]
y_test  = test[TARGET]

print(f'Train: {len(train)} rows ({train["date"].min().date()} → {train["date"].max().date()})')
print(f'Test:  {len(test)} rows ({test["date"].min().date()} → {test["date"].max().date()})')


### 4.4 Random Forest Model


In [ ]:
rf = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
r2  = r2_score(y_test, y_pred)
print(f'MAE: {mae:,.0f} bed nights')
print(f'R²:  {r2:.3f}')

# Actual vs. predicted
fig, ax = plt.subplots(figsize=(10, 4))
ax.scatter(y_test, y_pred, alpha=0.4, s=20)
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
ax.plot(lims, lims, 'r--', lw=1)
ax.set_xlabel('Actual nights_spent_nuts2')
ax.set_ylabel('Predicted')
ax.set_title(f'Random Forest — all countries  |  MAE={mae:,.0f}  R²={r2:.3f}')
plt.tight_layout()
plt.show()


### 4.5 SHAP Feature Importance


In [ ]:
# pip install shap  ←  run once if not installed
# !pip install shap -q
import shap

explainer  = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X_test)

# Global feature importance
shap.summary_plot(shap_values, X_test, plot_type='bar', show=True)

# Beeswarm — direction of effect
shap.summary_plot(shap_values, X_test, show=True)


### 4.6 Per-Country Performance


In [ ]:
# Break down MAE / R² per country on the test set
test_results = test[['date','country','city']].copy()
# Re-apply country dummies for test set lookup
test_results['y_true'] = y_test.values
test_results['y_pred'] = y_pred

# Malta was aggregated — map back
perf = []
for country in sorted(df['country'].unique()):
    mask = test_results['country'] == country
    if mask.sum() == 0:
        continue
    _mae = mean_absolute_error(test_results.loc[mask,'y_true'], test_results.loc[mask,'y_pred'])
    _r2  = r2_score(test_results.loc[mask,'y_true'], test_results.loc[mask,'y_pred'])
    perf.append({'Country': country, 'MAE': round(_mae), 'R²': round(_r2, 3), 'n_test': mask.sum()})

perf_df = pd.DataFrame(perf)
print(perf_df.to_string(index=False))

# Bar chart
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
perf_df.plot.bar(x='Country', y='MAE', ax=axes[0], legend=False, color='steelblue')
axes[0].set_title('MAE per country (test set)')
axes[0].set_ylabel('MAE (bed nights)')
perf_df.plot.bar(x='Country', y='R²', ax=axes[1], legend=False, color='seagreen')
axes[1].set_title('R² per country (test set)')
axes[1].axhline(0, color='red', lw=0.8, ls='--')
plt.tight_layout()
plt.show()


---
## 5. Graph Neural Network (GNN)

### 5.1 Why a GNN?

> **TODO:** Justify the GNN — viral tourism shocks don't respect national borders. A destination that goes viral (e.g. Albania via TikTok) can trigger spillover demand to neighbouring regions. A GNN allows us to model these spatial dependencies explicitly by representing destinations as nodes and shared tourist-flow corridors as edges, enabling risk propagation and failure prediction across the network.
>
> Describe graph construction: nodes = destinations, edges = shared origin markets / geographic proximity, edge weights = shared visitor nationality share.

In [ ]:
# ── GNN implementation ────────────────────────────────────────────────────────
# TODO: Implement graph construction + GNN training here.
# Suggested libraries: torch_geometric or spektral (Keras).
pass

---
## 6. Recommender System

### 6.1 Concept & Justification

> **TODO:** Explain the recommender system — what does it recommend, to whom, and why is it the right tool here? Possible framing: given a destination's current digital-signal profile, recommend which *resilience interventions* (capacity limits, diversification campaigns, off-season promotion) are most likely to dampen an incoming demand shock, based on similarity to historical shock-recovery patterns from other destinations.

In [ ]:
# ── Recommender System implementation ─────────────────────────────────────────
# TODO: Implement recommender logic here.
pass

---
## 7. Final Solution System Architecture

> **TODO (all):** Insert the end-to-end system architecture diagram showing how all components connect:
> Digital signals (GT / Wiki) → Feature engineering (seasonality + COVID controls) → Random Forest per region → GNN spillover layer → Recommender System → Resilience dashboard output.

<!-- ![Final architecture](images/final_architecture.png) -->

---
## 8. Conclusion

> **TODO (all):** Summarise main findings:
> - Do GT signals predict arrivals? With what lead time?
> - Which features matter most (SHAP)?
> - What do the GNN spillover results reveal?
> - How useful is the recommender for resilience planning?
>
> Then list limitations and future work.